# 🎬 Netflix Content Strategy & Catalog EDA
### An Executive Investigation into Streaming Economics, Licensing Lag, and Format Mechanics

**Author:** Senior Data Scientist & Python Architect  
**Stack:** Python 3.12, Pandas, NumPy, Plotly Express & Graph Objects  

---

## Executive Summary & Business Questions
1. **The Great Studio Defense Pivot:** How did Netflix transition its acquisition model from studio back-catalogs (Lag > 3 years) to proprietary Originals (Lag = 0)?
2. **Format Mechanics:** Why do movie runtimes cluster at 90–110 minutes, and why do TV shows face a statistical 'cliff' after Season 2?
3. **Global Localization:** How do target audience maturity tiers differ between key production powerhouses like the US, India, UK, and South Korea?
4. **Data Hygiene Rigor:** Proving the *Missing Director Hypothesis*—why missing directors in TV shows are a structural property of episodic production rather than corrupted data.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import plotly.io as pio

# Ensure project root is on sys.path
sys.path.append(os.path.abspath('..'))

from src.data_generator import ensure_data_exists
from src.data_cleaner import clean_netflix_data
from src.feature_engineer import engineer_features, explode_column
from src.visualizations import (
    plot_catalog_growth,
    plot_acquisition_lag_trend,
    plot_top_countries_audience,
    plot_genre_cooccurrence,
    plot_duration_distributions,
    plot_missing_directors_comparison
)

print("Modules successfully imported.")

## Phase 1: Data Ingestion & Fallback Engine
Load either the official Kaggle dataset or the automated synthetic fallback engine with 100% schema parity.

In [ ]:
data_path = ensure_data_exists('../data/netflix_titles.csv')
raw_df = pd.read_csv(data_path)
print(f"Raw dataset loaded: {raw_df.shape[0]:,} rows, {raw_df.shape[1]} columns.")
raw_df.head(3)

## Phase 2: Data Hygiene & Parsing Pipeline
Handling categorical imputation, dropping irrecoverable records, standardizing datetime fields, and parsing duration into numeric units.

In [ ]:
cleaned_df, report = clean_netflix_data(raw_df)
print(f"Initial rows: {report['initial_rows']:,}")
print(f"Cleaned rows: {report['final_rows']:,} (Dropped: {report['dropped_rows']} rows, {report['dropped_pct']}%)")

# Display Before vs After Missing Value Audit
b_df = report["missing_before_df"].rename(columns={"Missing_Count": "Raw_Count", "Missing_Pct": "Raw_%"})
a_df = report["missing_after_df"].rename(columns={"Missing_Count": "Cleaned_Count", "Missing_Pct": "Cleaned_%"})
audit_df = pd.merge(b_df, a_df, on="Column", how="outer").fillna(0)
audit_df

### Proving the 'Missing Director Hypothesis'
Notice the vast discrepancy in missing directors between Movies and TV Shows.

In [ ]:
h_fig = plot_missing_directors_comparison(report)
h_fig.show()

## Phase 3: Strategic Feature Engineering
Calculating acquisition lag, categorizing proxy acquisition models, and mapping MPAA/TV ratings to clean audience tiers.

In [ ]:
df = engineer_features(cleaned_df)
df[['title', 'type', 'release_year', 'year_added', 'acquisition_lag', 'content_origin_proxy', 'audience_tier']].head(5)

## Phase 4: Interactive Visual Explorations

### 1. Catalog Growth & Velocity (Movies vs. TV Shows)

In [ ]:
fig_growth = plot_catalog_growth(df)
fig_growth.show()

### 2. Studio Licensing Defense (Acquisition Lag Evolution)

In [ ]:
fig_lag = plot_acquisition_lag_trend(df)
fig_lag.show()

### 3. Global Production Hubs & Audience Tiers

In [ ]:
fig_countries = plot_top_countries_audience(df, top_n=12)
fig_countries.show()

### 4. Genre Co-Occurrence Matrix

In [ ]:
fig_genres = plot_genre_cooccurrence(df, top_n=10)
fig_genres.show()

### 5. Runtime Sweet Spot & The Season 2 Cliff

In [ ]:
fig_duration = plot_duration_distributions(df)
fig_duration.show()